# Evolving a Minimal Neural Agent for Capture-the-Flag

Google Colab notebook for the final project codebase.

This notebook is designed to run in Colab from the GitHub repository. It imports the project code, evaluates saved agents, generates simple plots, and gives you a safe place to run short experiments without editing the source files.

## 1. Setup

If this notebook is opened inside Colab without the repo files, the next cell clones the GitHub repository. If the repository is private and cloning fails, the cell will ask for a GitHub token. The token is only used for the clone command in that Colab runtime.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/Nkurosky/melting-pot-ctrnn-ctf.git'
REPO_DIR = Path('/content/melting-pot-ctrnn-ctf')

def run(cmd, **kwargs):
    return subprocess.run(cmd, check=True, text=True, **kwargs)

if not Path('toy_ctf.py').exists() and not REPO_DIR.exists():
    try:
        run(['git', 'clone', REPO_URL, str(REPO_DIR)])
    except subprocess.CalledProcessError:
        import getpass
        token = getpass.getpass('Repository may be private. Paste a GitHub token with repo access: ')
        private_url = REPO_URL.replace('https://', f'https://{token}@')
        run(['git', 'clone', private_url, str(REPO_DIR)])
        token = None

if REPO_DIR.exists():
    os.chdir(REPO_DIR)

run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
print('Working directory:', ROOT)

## 2. Import the model

The main controller and sensor utilities live in `toy_ctf.py`. The two-agent capture-the-flag environment lives in `mini_ctf.py`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import toy_ctf
import mini_ctf

print('Stage 1 sensor count:', toy_ctf.N_SENSORS)
print('Stage 1 genome params:', toy_ctf.N_PARAMS)
print('Mini-CTF sensor count:', mini_ctf.N_SENSORS)
print('Mini-CTF genome params:', mini_ctf.N_PARAMS)

## 3. Evaluate the saved Stage 1 champion

This reproduces the paper-style comparison between the saved Stage 1 champion and random controllers. It uses held-out random seeds so the score is not just one lucky episode.

In [ ]:
rng = np.random.default_rng(123)
stage1_path = Path('best_genome.npy')

def eval_stage1(genome, seeds):
    fits, caps = [], []
    for seed in seeds:
        fit, captures, _ = toy_ctf.run_episode(genome, seed=int(seed), record=False)
        fits.append(fit)
        caps.append(captures)
    return np.array(fits), np.array(caps)

seeds = np.arange(1000, 1050)

if stage1_path.exists():
    champion = np.load(stage1_path)
else:
    print('No saved best_genome.npy found; evolving a quick Stage 1 champion instead.')
    champion, _ = toy_ctf.evolve(pop=20, gens=10, trials=2, seed=0, verbose=False)

champ_fit, champ_caps = eval_stage1(champion, seeds)
random_genomes = rng.normal(0, 1.0, size=(20, toy_ctf.N_PARAMS))
random_stats = [eval_stage1(g, seeds[:10]) for g in random_genomes]
random_fit_means = np.array([x[0].mean() for x in random_stats])
random_cap_means = np.array([x[1].mean() for x in random_stats])

print(f'Champion mean fitness: {champ_fit.mean():.1f} +/- {champ_fit.std():.1f}')
print(f'Champion mean captures: {champ_caps.mean():.2f} +/- {champ_caps.std():.2f}')
print(f'Random mean fitness: {random_fit_means.mean():.1f} +/- {random_fit_means.std():.1f}')
print(f'Random mean captures: {random_cap_means.mean():.3f} +/- {random_cap_means.std():.3f}')

In [ ]:
labels = ['Random baseline', 'Stage 1 champion']
fit_means = [random_fit_means.mean(), champ_fit.mean()]
fit_errs = [random_fit_means.std(), champ_fit.std()]
cap_means = [random_cap_means.mean(), champ_caps.mean()]
cap_errs = [random_cap_means.std(), champ_caps.std()]

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].bar(labels, fit_means, yerr=fit_errs, capsize=4)
axes[0].set_ylabel('Mean fitness')
axes[0].set_title('Fitness')
axes[1].bar(labels, cap_means, yerr=cap_errs, capsize=4)
axes[1].set_ylabel('Mean captures')
axes[1].set_title('Captures')
plt.suptitle('Stage 1 champion vs. random controllers')
plt.tight_layout()
plt.show()

## 4. Run and visualize one mini-CTF match

This cell does not train. It just runs one match between two random genomes so you can inspect what the simulator records.

In [ ]:
rng = np.random.default_rng(0)
genome_a = rng.normal(size=mini_ctf.N_PARAMS)
genome_b = rng.normal(size=mini_ctf.N_PARAMS)

fit_a, fit_b, winner, trail = mini_ctf.run_match(genome_a, genome_b, seed=1, record=True)
print('fit_a:', round(fit_a, 1), 'fit_b:', round(fit_b, 1), 'winner:', winner, 'frames:', len(trail))

arr = np.array(trail)
plt.figure(figsize=(6, 6))
plt.xlim(0, mini_ctf.ARENA)
plt.ylim(0, mini_ctf.ARENA)
plt.gca().set_aspect('equal')
plt.plot(arr[:, 0], arr[:, 1], label='Agent A')
plt.plot(arr[:, 4], arr[:, 5], label='Agent B')
plt.scatter([mini_ctf.base_pos(0)[0], mini_ctf.base_pos(1)[0]], [mini_ctf.base_pos(0)[1], mini_ctf.base_pos(1)[1]], marker='s', label='Bases')
plt.title(f'Mini-CTF random match paths, winner={winner}')
plt.legend()
plt.show()

## 5. Paper figures from the repo

The committed paper figures are displayed below if they are present. These are the figures referenced in `docs/final_paper_draft.md`.

In [ ]:
from IPython.display import Image, display

for path in [
    'docs/figures/stage1_training_curve.png',
    'docs/figures/stage1_champion_comparison.png',
    'docs/figures/mini_ctf_100gen_summary.png',
]:
    p = Path(path)
    if p.exists():
        print(path)
        display(Image(filename=str(p)))
    else:
        print('Missing:', path)

## 6. Optional: quick evolution run

This is intentionally off by default. Set `RUN_QUICK_EVOLUTION = True` if you want Colab to run a short mini-CTF evolution pass. Longer paper-quality runs are better from the command line because they can take several minutes or more.

In [ ]:
RUN_QUICK_EVOLUTION = False

if RUN_QUICK_EVOLUTION:
    best, fits, wrs, hof, alltime_best, alltime_fit = mini_ctf.evolve(pop=16, gens=10, seed=7)
    plt.figure(figsize=(7, 4))
    plt.plot(fits, label='best fitness')
    plt.plot(wrs, label='best win rate')
    plt.legend()
    plt.title(f'Quick mini-CTF run, all-time fit={alltime_fit:.1f}')
    plt.show()
else:
    print('Skipping quick evolution. Set RUN_QUICK_EVOLUTION = True to run it.')

## 7. Notes for the final paper

Use this section to record observations from your own runs.

- What behaviors do you see in replay?
- Does the agent pick up the flag?
- Does it return home after pickup?
- Does it get stuck in spinning, wall-hugging, or chasing the wrong object?
- Does the all-time best behave better than the final-generation best?